# DSML 4220 - Lab 9: Finetuning GPT-2 with LoRA

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sgeinitz/DSML4220/blob/main/lab9_finetuning_gpt2.ipynb)

[![Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/sgeinitz/DSML4220/blob/main/lab9_finetuning_gpt2.ipynb)

In this lab we will use GPT-2 for the task of text generation. We'll first quickly compare Greedy Search and (Diverse) Beam Search with GPT-2. Then we'll finetune GPT-2 to generate text that is more explicitly infused with knowledge of Hemingway's book, "_The Sun also Rises_", and can generate text in the style of the book.


### Lab 9 Assignment/Task
There are three questions in this lab. As an added bonus, try downloading your own book from Project Gutenberg to finetune GPT-2 to generate text following your chosen book/author (see [this script](https://github.com/sgeinitz/DSML4220/blob/main/misc04_convert_book2csv.py)) for help to convert it to a .csv file of sentences).

In [15]:
import torch

import numpy as np
import pandas as pd

from transformers import GPT2Tokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from torch.utils.data import Dataset, random_split
from peft import get_peft_model, LoraConfig

In [16]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("openai-community/gpt2")

In [17]:
inputs = tokenizer(["Today is"], return_tensors="pt")
inputs

{'input_ids': tensor([[8888,  318]]), 'attention_mask': tensor([[1, 1]])}

Let's generate some text from the model using regular Greedy Search (here is the [HuggingFace example documenting this](https://huggingface.co/docs/transformers/en/main_classes/text_generation#transformers.GenerationMixin.compute_transition_scores.example)).

In [18]:
# Example 1: Print the scores for each token generated with Greedy Search
#tokenizer.pad_token_id = tokenizer.eso_token_id
outputs = model.generate(**inputs, max_new_tokens=20, return_dict_in_generate=True, output_scores=True)
transition_scores = model.compute_transition_scores(
    outputs.sequences, outputs.scores, normalize_logits=True
)
# input_length is the length of the input prompt for decoder-only models, like the GPT family, and 1 for
# encoder-decoder models, like BART or T5.
input_length = 1 if model.config.is_encoder_decoder else inputs.input_ids.shape[1]
generated_tokens = outputs.sequences[:, input_length:]
for tok, score in zip(generated_tokens[0], transition_scores[0]):
    # | token | token string | log probability | probability
    print(f"| {tok:5d} | {tokenizer.decode(tok):8s} | {score.numpy():.3f} | {np.exp(score.numpy()):.2%}")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


|   262 |  the     | -1.414 | 24.33%
|  1110 |  day     | -2.609 | 7.36%
|   618 |  when    | -2.010 | 13.41%
|   356 |  we      | -1.859 | 15.58%
|   460 |  can     | -2.508 | 8.14%
|   477 |  all     | -2.752 | 6.38%
|   307 |  be      | -2.960 | 5.18%
|  6613 |  proud   | -2.135 | 11.82%
|   286 |  of      | -0.558 | 57.21%
|   674 |  our     | -1.472 | 22.96%
|  1499 |  country | -2.392 | 9.15%
|   290 |  and     | -1.474 | 22.90%
|   674 |  our     | -2.127 | 11.92%
|  3815 |  values  | -2.251 | 10.53%
|    13 | .        | -1.281 | 27.78%
|   198 | 
        | -1.595 | 20.30%
|   198 | 
        | -0.000 | 99.96%
|  1135 | We       | -2.370 | 9.35%
|   389 |  are     | -1.931 | 14.50%
|   477 |  all     | -2.258 | 10.46%


In [19]:
outputs['sequences']

tensor([[8888,  318,  262, 1110,  618,  356,  460,  477,  307, 6613,  286,  674,
         1499,  290,  674, 3815,   13,  198,  198, 1135,  389,  477]])

Let's now use Beam Search (again using [this example from HF](https://huggingface.co/docs/transformers/en/main_classes/text_generation#transformers.GenerationMixin.compute_transition_scores.example)).

In [20]:
inputs = tokenizer(["Today is"], return_tensors="pt")

# Approach 2: Beam Search
outputs = model.generate(
    **inputs,
    max_new_tokens=10,
    num_beams=6,
    num_beam_groups=3,
    diversity_penalty=5.0,
    num_return_sequences=6,
    return_dict_in_generate=True,
    output_scores=True,
)
transition_scores = model.compute_transition_scores(
    outputs.sequences, outputs.scores, outputs.beam_indices, normalize_logits=False
)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In [21]:
outputs['sequences']

tensor([[8888,  318,  407,  262,  886,  286,  262,  995,   11,  475,  340,  318],
        [8888,  318,  407,  262,  886,  286,  262,  995,   11,  475,  340,  338],
        [8888,  318,  262,  640,  329,  514,  284, 1011,  257, 1302, 1028,  262],
        [8888,  318,  262,  640,  329,  514,  284, 1011,  257, 1302, 1028,  428],
        [8888,  318,  618,  345,  423,  257, 1256,  286,  640,  284,  892,  546],
        [8888,  318,  618,  345,  423,  257, 1256,  286,  640,  284,  892,   13]])

In [22]:
for s, seq in enumerate(outputs['sequences']):
  print(f"seq {s}: {tokenizer.decode(seq)}")

seq 0: Today is not the end of the world, but it is
seq 1: Today is not the end of the world, but it's
seq 2: Today is the time for us to take a stand against the
seq 3: Today is the time for us to take a stand against this
seq 4: Today is when you have a lot of time to think about
seq 5: Today is when you have a lot of time to think.


---

### Q1: Does the Beam Search above use Diverse Beam Search? If not, change it to use Diverse Beam Search and describe how the output differs.  

(Hint: Look a few cells down at the next use of Beam Search, there are two parameters you will need to add, `num_beam_groups`, and `diversity_penalty`)

`No, it currently does not. With the regular beam search the 6 returned sequences were essentially identical. With the diverse beam search enabled onle 1 sequence was returned, which is different in structure and content. `

---

In [23]:
prompt = ["Cohn confronted the bullfighter and "]
inputs = tokenizer(prompt, return_tensors="pt")

max_new_toks = 20
# Example 1: Print the scores for each token generated with Greedy Search
#outputs = model.generate(**inputs, max_new_tokens=max_new_toks, return_dict_in_generate=True, output_scores=True, do_sample=True, temperature=1)
outputs = model.generate(**inputs, max_new_tokens=max_new_toks, return_dict_in_generate=True, output_scores=True, do_sample=False)
transition_scores = model.compute_transition_scores(
    outputs.sequences, outputs.scores, normalize_logits=True
)
# input_length is the length of the input prompt for decoder-only models, like the GPT family, and 1 for
# encoder-decoder models, like BART or T5.
input_length = 1 if model.config.is_encoder_decoder else inputs.input_ids.shape[1]
generated_tokens = outputs.sequences[:, input_length:]
for tok, score in zip(generated_tokens[0], transition_scores[0]):
    # | token | token string | log probability | probability
    print(f"| {tok:5d} | {tokenizer.decode(tok):8s} | {score.numpy():.3f} | {np.exp(score.numpy()):.2%}")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


|  3711 | iced     | -0.646 | 52.43%
|   340 |  it      | -1.618 | 19.82%
|   510 |  up      | -0.896 | 40.80%
|    13 | .        | -1.199 | 30.16%
|   198 | 
        | -1.543 | 21.38%
|   198 | 
        | -0.018 | 98.22%
|     1 | "        | -0.677 | 50.79%
|    40 | I        | -1.864 | 15.50%
|  1101 | 'm       | -2.006 | 13.45%
|   407 |  not     | -1.525 | 21.76%
|  1016 |  going   | -1.388 | 24.95%
|   284 |  to      | -0.038 | 96.27%
|  1309 |  let     | -2.831 | 5.90%
|   345 |  you     | -0.957 | 38.42%
|   651 |  get     | -2.149 | 11.66%
|  1497 |  away    | -1.149 | 31.69%
|   351 |  with    | -0.175 | 83.95%
|   340 |  it      | -1.441 | 23.67%
|   553 | ,"       | -0.650 | 52.20%
|   339 |  he      | -1.260 | 28.36%


In [24]:
inputs = tokenizer(prompt, return_tensors="pt")

# Approach 2: Reconstruct the sequence scores from Beam Search
outputs = model.generate(
    **inputs,
    max_new_tokens=max_new_toks,
    num_beams=6,
    num_beam_groups=3,
    diversity_penalty=5.0,
    num_return_sequences=6,
    return_dict_in_generate=True,
    output_scores=True,
    temperature=2.0,
    #do_sample=True
)
transition_scores = model.compute_transition_scores(
    outputs.sequences, outputs.scores, outputs.beam_indices, normalize_logits=False
)
# If you sum the generated tokens' scores and apply the length penalty, you'll get the sequence scores.
# Tip 1: recomputing the scores is only guaranteed to match with `normalize_logits=False`. Depending on the
# use case, you might want to recompute it with `normalize_logits=True`.
# Tip 2: the output length does NOT include the input length
output_length = np.sum(transition_scores.numpy() < 0, axis=1)
length_penalty = 1.0 #model.generation_config.length_penalty
reconstructed_scores = transition_scores.sum(axis=1) / (output_length**length_penalty)

print(np.allclose(outputs.sequences_scores, reconstructed_scores))

for s, seq in enumerate(outputs['sequences']):
  print(f"seq {s}: {tokenizer.decode(seq)}")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


False
seq 0: Cohn confronted the bullfighter and iced him up.

"I'm not going to let you get away with it," he
seq 1: Cohn confronted the bullfighter and iced him up.

"I'm not going to let you get away with that," he
seq 2: Cohn confronted the bullfighter and ichthyologist, who had been working on the case for more than a year.

"
seq 3: Cohn confronted the bullfighter and ichthyologist, who had been working on the case for more than a year.

The
seq 4: Cohn confronted the bullfighter and urchin, who had been in a state of shock.

"I'm not going to
seq 5: Cohn confronted the bullfighter and urchin, who had been in a state of shock.

"You're not going to


C:\Users\joshl\AppData\Local\Temp\ipykernel_24152\629264205.py:25: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  reconstructed_scores = transition_scores.sum(axis=1) / (output_length**length_penalty)


Let's now load the raw text from Hemingway's book, _"The Sun also Rises"_.

In [25]:
heming = pd.read_csv("https://raw.githubusercontent.com/sgeinitz/DSML4220/main/data/sunalsorises.csv")
heming.head()

,sentence
0,Robert Cohn was once middleweight boxing champ...
1,Do not think that I am very much impressed by ...
2,"He cared nothing for boxing, in fact he dislik..."
3,There was a certain inner comfort in knowing h...
4,He was Spider Kelly’s star pupil.


In [26]:
sentences = heming['sentence']
sentences.head()

0    Robert Cohn was once middleweight boxing champ...
1    Do not think that I am very much impressed by ...
2    He cared nothing for boxing, in fact he dislik...
3    There was a certain inner comfort in knowing h...
4                    He was Spider Kelly’s star pupil.
Name: sentence, dtype: str

In [27]:
print(f"        sentence: '{sentences[0]}' \n is tokenized as: {tokenizer.encode(sentences[0])}")

        sentence: 'Robert Cohn was once middleweight boxing champion of Princeton.' 
 is tokenized as: [19156, 45005, 373, 1752, 3504, 6551, 21576, 8783, 286, 23173, 13]


In [28]:
max_length = max([len(tokenizer.encode(sentence)) for sentence in sentences])
max_length

224

In [29]:
class HemingwayDataset(Dataset):
    def __init__(self, txt_list, tokenizer, max_length):
        self.input_ids = []
        self.attn_masks = []
        self.labels = []
        for txt in txt_list:
            encodings_dict = tokenizer('<|startoftext|>' + txt + '<|endoftext|>', truncation=True,
                                       max_length=max_length, padding="max_length")
            self.input_ids.append(torch.tensor(encodings_dict['input_ids']))
            self.attn_masks.append(torch.tensor(encodings_dict['attention_mask']))

    def __len__(self): # overload the len() Python built-in function
        return len(self.input_ids)

    def __getitem__(self, idx): # overload the [] operator
        return self.input_ids[idx], self.attn_masks[idx]

tokenizer.pad_token_id = tokenizer.eos_token_id

In [30]:
dataset = HemingwayDataset(sentences, tokenizer, max_length=max_length)
train_size = int(0.9 * len(dataset))
train_dataset, val_dataset = random_split(dataset, [train_size, len(dataset) - train_size])

In [31]:
train_dataset[0]

(tensor([   27,    91,  9688,  1659,  5239,    91,    29,   447,   251,   564,
           250,    40,   447,   247,   303,  1392,   284,   466,  1223,    13,
         50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256, 50256, 50256, 50256, 50256, 5

Notice that above we set the `pad_token_id` to be the same as the `eos_token_id` (i.e. end-of-stream token id). So all of those `50256` entries above are being used as end-of-stream, or end-of-sequence tokens (except the first one, which is denoting the end of the sequence).  

In [32]:
tokenizer.decode([50256])

'<|endoftext|>'

In [33]:
batch_size = 16
n_epochs = 3
training_args = TrainingArguments(output_dir='~/hemingway_generation', num_train_epochs=n_epochs, logging_steps=100, save_steps=500, do_eval=True,
                                  eval_steps=20, per_device_train_batch_size=batch_size, per_device_eval_batch_size=batch_size,
                                  warmup_steps=10, weight_decay=0.05, logging_dir='~/hemingway_generation/logs', report_to='none')

Let's load GPT-2 and then  take a rough glance at the architecture of GPT (w/ ~130M parameters) by printing the model.

In [34]:
model = AutoModelForCausalLM.from_pretrained("openai-community/gpt2")
print(model)

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)


In [35]:
def count_trainable_parameters(mod):
    model_parameters = filter(lambda p: p.requires_grad, mod.parameters())
    params = sum([np.prod(p.size()) for p in model_parameters])
    return params

gpt2_params = count_trainable_parameters(model)
print(f"GPT-2 trainable parameters: {gpt2_params}")

GPT-2 trainable parameters: 124439808


In [36]:
trainer = Trainer(model=model,  args=training_args, train_dataset=train_dataset, eval_dataset=val_dataset,
                  data_collator=lambda data: {'input_ids': torch.stack([f[0] for f in data]),
                                              'attention_mask': torch.stack([f[1] for f in data]),
                                              'labels': torch.stack([f[0] for f in data])})

# on Colab this will take 6+hrs w/ cpu or <10min w/ T4 GPU per epoch
trainer.train()

`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
100,0.955700
200,0.211200
300,0.199600
400,0.199100
500,0.183600
600,0.177200
700,0.182200
800,0.179400
900,0.172000
1000,0.173000


TrainOutput(global_step=1152, training_loss=0.2505211118194792, metrics={'train_runtime': 409.6757, 'train_samples_per_second': 44.984, 'train_steps_per_second': 2.812, 'total_flos': 2106716000256000.0, 'train_loss': 0.2505211118194792, 'epoch': 3.0})

In [37]:
cuda = False
if torch.cuda.is_available():
    cuda = True
device = torch.device("cuda" if cuda else "cpu")
print("Using CUDA: {}".format(cuda))

Using CUDA: True


In [41]:
outputs = model.generate(
    **inputs,
    max_new_tokens=max_new_toks,
    num_beams=6,
    num_beam_groups=3,
    diversity_penalty=5.0,
    num_return_sequences=6,
    return_dict_in_generate=True,
    output_scores=True,
    temperature=2.0,
    #do_sample=True
)
#transition_scores = model.compute_transition_scores(
#    outputs.sequences, outputs.scores, outputs.beam_indices, normalize_logits=False
#)

for s, seq in enumerate(outputs['sequences']):
    gen_text = tokenizer.decode(seq)
    gen_text = gen_text[:gen_text.find('<|endoftext|>')]
    print(f"seq {s}: {gen_text}")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


seq 0: Cohn confronted the bullfighter and ’s owner.
seq 1: Cohn confronted the bullfighter and iced him.
seq 2: Cohn confronted the bullfighter and iced him with a bottle.
seq 3: Cohn confronted the bullfighter and ��’” “He’s a good bull-fighter,” h
seq 4: Cohn confronted the bullfighter and ��’” “He’s a good bull-fighter,” 
seq 5: Cohn confronted the bullfighter and ’s owner, who had been in the ring for three years, and the bull-fighte


Next, let's use LoRA to fine tune the model. We'll load the model again to ensure that the earlier finetuning is not included.

In [42]:
# load the model again so that we can use LoRA
model = AutoModelForCausalLM.from_pretrained("openai-community/gpt2")
print(model)

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)


In [43]:
lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    inference_mode=False,
    r=8,
    lora_alpha=32,
    lora_dropout=0.01,
    fan_in_fan_out=True
)

lora_model = get_peft_model(model, lora_config)#, "default")

W0428 13:35:07.791000 24152 torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [44]:
print(lora_model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): GPT2LMHeadModel(
      (transformer): GPT2Model(
        (wte): Embedding(50257, 768)
        (wpe): Embedding(1024, 768)
        (drop): Dropout(p=0.1, inplace=False)
        (h): ModuleList(
          (0-11): 12 x GPT2Block(
            (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (attn): GPT2Attention(
              (c_attn): lora.Linear(
                (base_layer): Conv1D(nf=2304, nx=768)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.01, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=768, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2304, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
          

In [45]:
trainer = Trainer(model=lora_model,  args=training_args, train_dataset=train_dataset, eval_dataset=val_dataset,
                  data_collator=lambda data: {'input_ids': torch.stack([f[0] for f in data]),
                                              'attention_mask': torch.stack([f[1] for f in data]),
                                              'labels': torch.stack([f[0] for f in data])})

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [46]:
trainer.train()

Step,Training Loss
100,8.240700
200,0.820800
300,0.462100
400,0.373800
500,0.307700
600,0.279400
700,0.276900
800,0.272100
900,0.267300
1000,0.264400


TrainOutput(global_step=1152, training_loss=1.0378750786185265, metrics={'train_runtime': 320.9431, 'train_samples_per_second': 57.421, 'train_steps_per_second': 3.589, 'total_flos': 2114020550541312.0, 'train_loss': 1.0378750786185265, 'epoch': 3.0})

---

### Q2: How many more `training_samples_per_second` could the LoRA model get through during finetuning than the original GPT-2 model could?

`The original GPT-2 model: train_samples_per_second': 44.984 compared to the LoRA model was 'train_samples_per_second': 57.421.`

---

In [47]:
inputs = tokenizer(prompt, return_tensors="pt").to(device)

# Use Diverse Beam Search
outputs = lora_model.generate(
    **inputs,
    max_new_tokens=max_new_toks,
    num_beams=6,
    num_beam_groups=3,
    diversity_penalty=5.0,
    num_return_sequences=6,
    return_dict_in_generate=True,
    output_scores=True,
    temperature=2.0,
    #do_sample=True
)
#transition_scores = lora_model.compute_transition_scores(
#    outputs.sequences, outputs.scores, outputs.beam_indices, normalize_logits=False
#)

#output_length = np.sum(transition_scores.cpu().numpy() < 0, axis=1)
#length_penalty = 1.0 #lora_model.generation_config.length_penalty
#reconstructed_scores = transition_scores.cpu().sum(axis=1).numpy() / (output_length**length_penalty)

for s, seq in enumerate(outputs['sequences']):
  gen_text = tokenizer.decode(seq)
  # remove everything from '<|endoftext|>' to the end from gen_text
  gen_text = gen_text[:gen_text.find('<|endoftext|>')]
  print(f"seq {s}: {gen_text}")

C:\Users\joshl\AppData\Roaming\Python\Python312\site-packages\transformers\generation\configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `2.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


seq 0: Cohn confronted the bullfighter and ________, and ________, ________, ________, ________, ________, _______
seq 1: Cohn confronted the bullfighter and iced it was a good fight.


seq 2: Cohn confronted the bullfighter and ________, and ________, ________, ________, ________, ________, and
seq 3: Cohn confronted the bullfighter and iced it was a good fight.
seq 4: Cohn confronted the bullfighter and ____. The bullfighter said, "You know, I'm not going to get into the figh
seq 5: Cohn confronted the bullfighter and ____. The bullfighter said, "You know, I'm not going to get into the busines


In [48]:
lora_params = count_trainable_parameters(lora_model)
print(f"LoRA trainable parameters: {lora_params} ({(100*lora_params/gpt2_params):.2f}% of GPT-2's trainable parameters)")

LoRA trainable parameters: 294912 (0.24% of GPT-2's trainable parameters)


---

### Q3: How many fewer parameters did the LoRA model need to train/tune than the full GPT-2 model did?

(Hint: See output from above cell)

`The LoRA model only needed to train 294,912 parameters, compared to GPT-2's 124 million parameters. So about 0.24% of the full model's trainable parameters.`

---